# Processamento da Camada Silver

## 1. Objetivo

Tratar e validar os dados provenientes da camada Bronze, elevando a qualidade, a consistência e a confiabilidade das informações antes de disponibilizá-las para análises na camada Gold.

## 2. Estratégia de processamento

O processamento é desenvolvido em PySpark, com regras implementadas em funções reutilizáveis no módulo `src/silver.py`.

As principais operações são:

- Padronização de campos textuais.
- Conversão e validação de tipos de dados.
- Validação matemática dos CPFs.
- Verificação de integridade referencial entre clientes, contas e transações.
- Deduplicação por chave de negócio.
- Separação dos registros inválidos em tabelas de quarentena, com o respectivo motivo de rejeição.

## 3. Estrutura de saída

**Tabelas Silver**
- `silver_clientes`
- `silver_contas`
- `silver_transacoes_pix`

**Tabelas de quarentena**
- `quarantine_clientes`
- `quarantine_contas`
- `quarantine_transacoes_pix`

O processamento segue a dependência entre as entidades: clientes → contas → transações PIX.

## 4. Tratamento de clientes

Os registros de `bronze_clientes` passam por limpeza dos campos textuais, conversão da data de nascimento, validação de CPF e deduplicação por `id_cliente`.

A validação do CPF considera o formato de 11 dígitos numéricos e o cálculo dos dígitos verificadores. Os registros são direcionados para:

- **Silver:** clientes aprovados nas regras de validação, sem duplicidade de chave.
- **Quarentena:** registros rejeitados, acompanhados do motivo da rejeição.

In [0]:
import importlib
import src.silver

importlib.reload(src.silver)

### 4.1. Execução do processamento

Execução das etapas de limpeza, validação, deduplicação e persistência das tabelas de clientes.

In [0]:
df_silver_clientes, df_quarantine_clientes = src.silver.process_clients(spark)

print("Clientes na Silver:", df_silver_clientes.count())
print("Clientes em quarentena:", df_quarantine_clientes.count())

### 4.2. Validação dos resultados

Conferência dos volumes, verificação de duplicidades por `id_cliente` e análise dos motivos de rejeição.

In [0]:
%sql
SELECT 'silver_clientes' AS tabela, COUNT(*) AS registros
FROM workspace.silver.silver_clientes
UNION ALL
SELECT 'quarantine_clientes', COUNT(*)
FROM workspace.silver.quarantine_clientes;

In [0]:
%sql
SELECT id_cliente, COUNT(*) AS quantidade
FROM workspace.silver.silver_clientes
GROUP BY id_cliente
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT motivo_rejeicao, COUNT(*) AS quantidade
FROM workspace.silver.quarantine_clientes
GROUP BY motivo_rejeicao
ORDER BY quantidade DESC;

## 5. Tratamento de contas

Os registros de `bronze_contas` passam pela padronização dos campos textuais e pela conversão da data de abertura.

Também é verificada a integridade referencial com `silver_clientes`, garantindo que cada conta válida esteja associada a um cliente existente na camada Silver. Registros inválidos são encaminhados para quarentena com o motivo correspondente.

### 5.1. Execução do processamento

Execução das etapas de limpeza, validação de integridade referencial, deduplicação e persistência das tabelas de contas.

In [0]:
df_silver_contas, df_quarantine_contas = src.silver.process_accounts(spark)

print("Contas na Silver:", df_silver_contas.count())
print("Contas em quarentena:", df_quarantine_contas.count())

display(df_silver_contas.limit(10))

### 5.2. Validação dos resultados

Conferência dos volumes, verificação de duplicidades por `id_conta`, validação da integridade referencial e análise dos motivos de rejeição.

In [0]:
%sql
SELECT 'silver_contas' AS tabela, COUNT(*) AS registros
FROM workspace.silver.silver_contas
UNION ALL
SELECT 'quarantine_contas', COUNT(*)
FROM workspace.silver.quarantine_contas;

In [0]:
%sql
SELECT id_conta, COUNT(*) AS quantidade
FROM workspace.silver.silver_contas
GROUP BY id_conta
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*) AS clientes_nao_encontrados
FROM workspace.silver.silver_contas a
LEFT JOIN workspace.silver.silver_clientes c
    ON a.id_cliente = c.id_cliente
WHERE c.id_cliente IS NULL;

In [0]:
%sql
SELECT motivo_rejeicao, COUNT(*) AS quantidade
FROM workspace.silver.quarantine_contas
GROUP BY motivo_rejeicao
ORDER BY quantidade DESC;

## 6. Tratamento de transações PIX

Os registros de `bronze_transacoes_pix` passam por padronização dos campos textuais e conversão dos identificadores, datas e valores monetários.

As regras de qualidade contemplam:

- Validação dos campos obrigatórios (`id_transacao`, `id_conta`, `data_transacao` e `valor`).
- Verificação da existência da conta associada em `silver_contas`.
- Deduplicação por `id_transacao`.
- Encaminhamento dos registros inválidos para quarentena, com o motivo da rejeição.

### 6.1. Execução do processamento

Execução integrada das etapas de limpeza, classificação, validação, deduplicação e persistência das tabelas de transações PIX.

In [0]:
df_silver_transacoes, df_quarantine_transacoes = src.silver.process_transactions(spark)

print("Transações na Silver:", df_silver_transacoes.count())
print("Transações em quarentena:", df_quarantine_transacoes.count())

display(df_silver_transacoes.limit(10))

### 6.2. Análise dos registros em quarentena

Agrupamento dos registros rejeitados por motivo para identificar as principais ocorrências de qualidade dos dados.

In [0]:
display(
    df_quarantine_transacoes
    .groupBy("motivo_rejeicao")
    .count()
    .orderBy("count", ascending=False)
)

### 6.3. Validação da qualidade dos dados

Verificação da ausência de duplicidades, de valores nulos nos campos obrigatórios e de referências a contas inexistentes na tabela Silver.

**Verificação de duplicidades por `id_transacao`.** O resultado esperado é não retornar registros.

In [0]:
%sql
SELECT id_transacao, COUNT(*) AS quantidade
FROM workspace.silver.silver_transacoes_pix
GROUP BY id_transacao
HAVING COUNT(*) > 1;

**Verificação de valores nulos nos campos obrigatórios.** O resultado esperado é zero.

In [0]:
%sql
SELECT COUNT(*) AS registros_invalidos
FROM workspace.silver.silver_transacoes_pix
WHERE id_transacao IS NULL
   OR id_conta IS NULL
   OR data_transacao IS NULL
   OR valor IS NULL;

**Verificação da integridade referencial com `silver_contas`.** O resultado esperado é zero.

In [0]:
%sql
SELECT COUNT(*) AS contas_nao_encontradas
FROM workspace.silver.silver_transacoes_pix t
LEFT JOIN workspace.silver.silver_contas c
    ON t.id_conta = c.id_conta
WHERE c.id_conta IS NULL;

## 7. Conferência geral dos volumes

Comparação dos registros de origem (Bronze) com os registros aprovados (Silver) e rejeitados (quarentena). A diferença corresponde aos registros excedentes removidos durante a deduplicação, conforme as regras aplicadas no processamento.

In [0]:
%sql
SELECT
    'Clientes' AS entidade,
    (SELECT COUNT(*) FROM workspace.bronze.bronze_clientes) AS bronze,
    (SELECT COUNT(*) FROM workspace.silver.silver_clientes) AS silver,
    (SELECT COUNT(*) FROM workspace.silver.quarantine_clientes) AS quarentena,
    (SELECT COUNT(*) FROM workspace.bronze.bronze_clientes)
      - (SELECT COUNT(*) FROM workspace.silver.silver_clientes)
      - (SELECT COUNT(*) FROM workspace.silver.quarantine_clientes) AS duplicados_removidos
UNION ALL
SELECT
    'Contas',
    (SELECT COUNT(*) FROM workspace.bronze.bronze_contas),
    (SELECT COUNT(*) FROM workspace.silver.silver_contas),
    (SELECT COUNT(*) FROM workspace.silver.quarantine_contas),
    (SELECT COUNT(*) FROM workspace.bronze.bronze_contas)
      - (SELECT COUNT(*) FROM workspace.silver.silver_contas)
      - (SELECT COUNT(*) FROM workspace.silver.quarantine_contas)
UNION ALL
SELECT
    'Transações PIX',
    (SELECT COUNT(*) FROM workspace.bronze.bronze_transacoes_pix),
    (SELECT COUNT(*) FROM workspace.silver.silver_transacoes_pix),
    (SELECT COUNT(*) FROM workspace.silver.quarantine_transacoes_pix),
    (SELECT COUNT(*) FROM workspace.bronze.bronze_transacoes_pix)
      - (SELECT COUNT(*) FROM workspace.silver.silver_transacoes_pix)
      - (SELECT COUNT(*) FROM workspace.silver.quarantine_transacoes_pix)
ORDER BY entidade;